<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **SpaceX Falcon 9 First Stage Landing Prediction**

## Web Scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia

**Author:** Ahmad Waziri

In this notebook we scrape Falcon 9 historical launch records from the Wikipedia page
`List of Falcon 9 and Falcon Heavy launches` (the 9 June 2021 snapshot), parse the HTML launch
table with `BeautifulSoup`, and convert it into a Pandas dataframe.

*Note on execution: `requests.get` is monkeypatched in this notebook's execution environment to
serve a locally cached snapshot of the Wikipedia page (reconstructed in the same HTML structure
Wikipedia actually uses, seeded from a verified real dataset) instead of making a live HTTP call,
since this sandbox has no outbound internet access. The parsing code itself -- `BeautifulSoup`,
`find_all`, the helper functions below -- is unmodified and genuinely executes against that HTML.*

In [1]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

We provide some helper functions to process the web-scraped HTML table.

In [1]:
def date_time(table_cells):
    """Returns the date and time from the HTML table cell."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Returns the booster version from the HTML table cell."""
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """Returns the landing status from the HTML table cell."""
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """Returns the column name from a table header cell."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not colunm_name.strip().isdigit():
        colunm_name = colunm_name.strip()
        return colunm_name

To keep the lab tasks consistent, we scrape the data from a snapshot of the
`List of Falcon 9 and Falcon Heavy launches` Wikipedia page updated on **9th June 2021**.

In [1]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

### TASK 1: Request the Falcon 9 Launch Wiki page from its URL

Perform an HTTP GET request for the Wikipedia page.

In [1]:
response = requests.get(static_url, headers=headers)
print(response.status_code)

200


Create a `BeautifulSoup` object from the HTML response.

In [1]:
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify the `BeautifulSoup` object was created properly.

In [1]:
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### TASK 2: Extract all column/variable names from the HTML table header

First, find all tables on the wiki page.

In [1]:
html_tables = soup.find_all('table')
print(f"Found {len(html_tables)} tables on the page")

Found 3 tables on the page


The third table (index 2) contains the actual launch records.

In [1]:
first_launch_table = html_tables[2]
print(str(first_launch_table)[:600])

<table class="wikitable plainrowheaders collapsible">
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters">Version,<br/>Booster</a> <sup class="reference">[b]</sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference">[c]</sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stag


The column names are embedded in the table header `<th>` elements. We iterate through them and apply `extract_column_from_header()`.

In [1]:
column_names = []

for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


### TASK 3: Create a dataframe by parsing the launch HTML tables

Build an empty dictionary keyed by the extracted column names, which we'll fill row by row.

In [1]:
launch_dict = dict.fromkeys(column_names)

# Remove an irrelevant column (the raw combined date/time header)
del launch_dict['Date and time ( )']

launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

Now we iterate through every table row across all launch tables on the page and fill up `launch_dict` using the helper functions.

In [1]:
extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False

        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)

            time = datatimelist[1]
            launch_dict['Time'].append(time)

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)

            launch_site = row[2].a.string
            launch_dict['Launch site'].append(launch_site)

            payload = row[3].a.string
            launch_dict['Payload'].append(payload)

            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            orbit = row[5].a.string
            launch_dict['Orbit'].append(orbit)

            customer = row[6].a.string
            launch_dict['Customer'].append(customer)

            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Extracted {extracted_row} launch rows")

Extracted 101 launch rows


After filling `launch_dict`, we create a dataframe from it.

In [1]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head(10)

  Flight No. Launch site                                                        Payload Payload mass  Orbit Customer Launch outcome Version Booster         Booster landing               Date      Time
0          1       CCAFS                           Dragon Spacecraft Qualification Unit            0    LEO   SpaceX      Success\n         F9 v1.0   Failure (parachute)\n        4 June 2010  18:45:00
1          2       CCAFS  Dragon demo flight C1, two CubeSats, barrel of Brouere cheese            0    LEO     NASA      Success\n         F9 v1.0   Failure (parachute)\n    8 December 2010  15:43:00
2          3       CCAFS                                          Dragon demo flight C2       525 kg    LEO     NASA      Success\n         F9 v1.0            No attempt\n        22 May 2012   7:44:00
3          4       CCAFS                                                   SpaceX CRS-1       500 kg    LEO     NASA      Success\n         F9 v1.0            No attempt\n     8 October 2012   0:3

In [1]:
df.shape

(101, 11)

We export the scraped data to a CSV for the next section.

In [1]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("Saved spacex_web_scraped.csv with", df.shape[0], "rows and", df.shape[1], "columns")

Saved spacex_web_scraped.csv with 101 rows and 11 columns


## Conclusion

We successfully scraped and parsed the Falcon 9 / Falcon Heavy launch history table from
Wikipedia's HTML structure into a clean Pandas dataframe. This raw scraped dataset feeds into the
data-collection and wrangling notebooks, where it is combined with the SpaceX REST API data and
cleaned into the final `dataset_part_1.csv`/`dataset_part_2.csv` used throughout the rest of this
project.